# Cross-Day Partial Unwinds Report

**Period:** March 1-10, 2026  
**Source:** DTCC SDR USD SOFR/FF swaps

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import sys, os

_nb_dir = os.path.dirname(os.path.abspath("__file__"))
_project_root = os.path.normpath(os.path.join(_nb_dir, "..", ".."))
for p in [_nb_dir, _project_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

import datetime
import pandas as pd
import _usd_swaps_common as sdr

sdr.notebook_setup()

In [ ]:
from SDRUtils.analytics.trade_tape import TradeTape

df = sdr.load_usd_swaps(
    datetime.datetime(2026, 3, 1),
    datetime.datetime(2026, 3, 10),
)
tape = TradeTape(df)
enriched = tape.compute()
print(f"Raw: {len(df):,} trades | Enriched: {len(enriched.columns)} columns")
tape.summary()

## Cross-Day Lifecycle Resolution

Now load with `raw_df` to enable cross-day lifecycle resolution.

In [ ]:
from SDRUtils.core.lifecycle import resolve_lifecycle_cross_day

classified_df, raw_df = sdr.load_usd_swaps(
    datetime.datetime(2026, 3, 1),
    datetime.datetime(2026, 3, 11),  # extend to capture full Mar 10 events
    return_raw=True,
)
print(f"Classified: {len(classified_df):,} | Raw (unfiltered): {len(raw_df):,}")

In [ ]:
# Run cross-day resolver on all classified trades
classified_ids = set(classified_df["trade_id"].astype(str))
xd = resolve_lifecycle_cross_day(raw_df, classified_ids, skip_intraday_only=True)
print(f"Cross-day resolved: {len(xd):,} trades")
print()
print("Status distribution:")
print(xd["xd_status"].value_counts())

## Partial Unwinds

In [ ]:
# Filter to partial unwinds
partial = xd[xd["xd_has_partial_unwind"] == True].copy()
print(f"Partial unwinds: {len(partial)}")
print()

if not partial.empty:
    # Merge with classified data for context
    partial.index.name = "trade_id"
    partial_merged = partial.reset_index().merge(
        classified_df[["trade_id", "tenor_label", "fixed_rate", "notional", "product_type", "execution_timestamp"]],
        on="trade_id",
        how="left",
    )
    
    display_cols = [
        "trade_id", "tenor_label", "product_type",
        "xd_inception_notional", "xd_current_notional", 
        "xd_notional_pct_remaining", "xd_status",
        "xd_n_events", "xd_n_days_spanned",
        "xd_is_terminated", "xd_fields_changed",
    ]
    available = [c for c in display_cols if c in partial_merged.columns]
    
    print("Partial unwind details:")
    display(partial_merged[available].sort_values("xd_notional_pct_remaining"))
else:
    print("No partial unwinds found in this date range.")

In [ ]:
# Summary statistics
if not partial.empty:
    print("=== Partial Unwind Summary (Mar 1-10, 2026) ===")
    print(f"Total partial unwinds: {len(partial)}")
    print(f"  Still active: {(partial['xd_status'] == 'PARTIAL_UNWIND').sum()}")
    print(f"  Subsequently terminated: {(partial['xd_status'] == 'TERMINATED').sum()}")
    print()
    print(f"Notional remaining distribution:")
    print(partial["xd_notional_pct_remaining"].describe())
    print()
    print(f"Days spanned distribution:")
    print(partial["xd_n_days_spanned"].value_counts().sort_index())
    print()
    print(f"Events per trade distribution:")
    print(partial["xd_n_events"].describe())